# 05 · Evaluación comparativa

**Responsable:** Julia Pérez

Este notebook **no entrena nada**: lee las predicciones que ha guardado cada compañera en
`resultados/` y las evalúa todas de la misma forma (sección 4.4 del enunciado):

1. Métricas para un problema multiclase desbalanceado.
2. Comparación de representaciones y modelos (nivel 2).
3. Incertidumbre del test: ¿qué diferencias son de verdad?
4. Repetición con nivel 1 (6 clases): ¿cambia el ranking?
5. Análisis de errores: F1 por clase, matriz de confusión y si las confusiones son entre categorías hermanas.

Formato que espera de cada modelo: `resultados/<modelo>_<nivel>_<conjunto>.csv`
con columnas `projectID, y_true, y_pred`. Por ejemplo `tfidf_logreg_nivel2_test.csv`.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

if os.path.exists("/content"):
    RUTA_DATOS = "/content/"
    RUTA_RESULTADOS = "/content/"
    RUTA_FIGURAS = "/content/"
else:
    RUTA_DATOS = "../data/"
    RUTA_RESULTADOS = "../resultados/"
    RUTA_FIGURAS = "../figuras/"

In [ ]:
# Datos con las etiquetas (generados en el notebook 01) y la partición común
datos = pd.read_pickle(RUTA_DATOS + "proyectos_etiquetados.pkl")
particion = pd.read_csv(RUTA_RESULTADOS + "particion.csv")
datos = datos.merge(particion, on="projectID")

# Cada categoría de nivel 2 cuelga de una única categoría de nivel 1 (su "madre")
madre = datos.groupby("nivel2")["nivel1"].first().to_dict()

# Categorías descartadas de cada proyecto (las otras áreas de nivel 2 a las que pertenece)
descartadas = dict(zip(datos["projectID"], datos["descartadas"]))

## 1. Métricas

- **Accuracy**: fracción de aciertos. La damos porque es intuitiva, pero con clases desbalanceadas
  engaña: el baseline de la clase mayoritaria ya tiene un 10,8 % en nivel 2 y un 39,2 % en nivel 1.
- **Macro-F1** (métrica principal): media del F1 de cada clase, **todas con el mismo peso**.
  Una clase de 9 proyectos cuenta igual que una de 3.464, así que un modelo solo puede tener un
  macro-F1 alto si funciona también en las clases pequeñas.
- **F1 por clase**: para ver en qué áreas funciona bien y en cuáles no.

Las decisiones (qué modelo es el mejor) se toman con **validación**. El test se usa para dar el
resultado final y su incertidumbre.

In [ ]:
def metricas(df):
    """Accuracy y macro-F1 de un DataFrame con columnas y_true e y_pred."""
    return {
        "accuracy": accuracy_score(df["y_true"], df["y_pred"]),
        "macro-F1": f1_score(df["y_true"], df["y_pred"], average="macro"),
    }

In [ ]:
def cargar_predicciones(nivel, conjunto):
    """Lee todos los ficheros <modelo>_<nivel>_<conjunto>.csv de resultados/."""
    predicciones = {}
    for fichero in sorted(glob.glob(RUTA_RESULTADOS + f"*_{nivel}_{conjunto}.csv")):
        modelo = os.path.basename(fichero).replace(f"_{nivel}_{conjunto}.csv", "")
        predicciones[modelo] = pd.read_csv(fichero)

    # Añadimos el baseline (clase mayoritaria de train) para tenerlo siempre de referencia
    clase_mayoritaria = datos.loc[datos["conjunto"] == "train", nivel].value_counts().index[0]
    sub = datos[datos["conjunto"] == conjunto]
    predicciones["baseline"] = pd.DataFrame({"projectID": sub["projectID"],
                                             "y_true": sub[nivel],
                                             "y_pred": clase_mayoritaria})
    return predicciones

pred_val = cargar_predicciones("nivel2", "val")
pred_test = cargar_predicciones("nivel2", "test")
print("Modelos encontrados:", list(pred_test.keys()))

## 2. Comparación de representaciones y modelos (nivel 2)

In [ ]:
filas = []
for modelo in pred_test:
    fila = {"modelo": modelo}
    if modelo in pred_val:
        m = metricas(pred_val[modelo])
        fila["val accuracy"], fila["val macro-F1"] = m["accuracy"], m["macro-F1"]
    m = metricas(pred_test[modelo])
    fila["test accuracy"], fila["test macro-F1"] = m["accuracy"], m["macro-F1"]
    filas.append(fila)

tabla_nivel2 = pd.DataFrame(filas).set_index("modelo").sort_values("val macro-F1", ascending=False).round(3)
tabla_nivel2.to_csv(RUTA_RESULTADOS + "evaluacion_nivel2.csv")
tabla_nivel2

In [ ]:
# El mejor modelo se elige con VALIDACIÓN, no con test
mejor = tabla_nivel2["val macro-F1"].idxmax()
print("Mejor modelo (según macro-F1 en validación):", mejor)

## 3. Incertidumbre del test

El test tiene 4.798 proyectos: si cogiéramos otros 4.798, las métricas cambiarían un poco.
Para medir cuánto usamos **bootstrap**: remuestreamos el test con reemplazamiento muchas veces,
recalculamos la métrica cada vez y nos quedamos con el intervalo que contiene el 95 % de los valores.

Para comparar dos modelos usamos **bootstrap emparejado**: en cada remuestreo cogemos los mismos
proyectos para los dos modelos y calculamos la diferencia. Si el intervalo de la diferencia
contiene el 0, **no podemos decir que un modelo sea mejor que el otro**.

In [ ]:
N_BOOT = 1000
rng = np.random.default_rng(42)
n_test = len(pred_test["baseline"])
remuestreos = [rng.integers(0, n_test, n_test) for _ in range(N_BOOT)]   # los mismos para todos

def bootstrap_macro_f1(df):
    """Macro-F1 en cada remuestreo bootstrap del test."""
    df = df.sort_values("projectID").reset_index(drop=True)   # mismo orden en todos los modelos
    y_true, y_pred = df["y_true"].values, df["y_pred"].values
    return np.array([f1_score(y_true[i], y_pred[i], average="macro") for i in remuestreos])

boot = {modelo: bootstrap_macro_f1(df) for modelo, df in pred_test.items()}

intervalos = pd.DataFrame({
    "test macro-F1": [metricas(pred_test[m])["macro-F1"] for m in boot],
    "IC 2,5 %": [np.percentile(boot[m], 2.5) for m in boot],
    "IC 97,5 %": [np.percentile(boot[m], 97.5) for m in boot],
}, index=list(boot)).round(3)
intervalos["semiancho"] = ((intervalos["IC 97,5 %"] - intervalos["IC 2,5 %"]) / 2).round(3)
intervalos.sort_values("test macro-F1", ascending=False)

In [ ]:
# Diferencia entre el mejor modelo y cada uno de los demás (bootstrap emparejado)
filas = []
for modelo in boot:
    if modelo == mejor:
        continue
    dif = boot[mejor] - boot[modelo]
    ic = np.percentile(dif, [2.5, 97.5])
    filas.append({"comparación": f"{mejor} - {modelo}",
                  "diferencia": metricas(pred_test[mejor])["macro-F1"] - metricas(pred_test[modelo])["macro-F1"],
                  "IC 2,5 %": ic[0], "IC 97,5 %": ic[1],
                  "¿significativa?": "sí" if ic[0] > 0 else "no"})

diferencias = pd.DataFrame(filas).round(3)
diferencias.to_csv(RUTA_RESULTADOS + "evaluacion_diferencias.csv", index=False)
diferencias

In [ ]:
# Gráfica: macro-F1 en test con su intervalo de confianza
orden = intervalos.sort_values("test macro-F1").index
err = [intervalos.loc[orden, "test macro-F1"] - intervalos.loc[orden, "IC 2,5 %"],
       intervalos.loc[orden, "IC 97,5 %"] - intervalos.loc[orden, "test macro-F1"]]

plt.figure(figsize=(7, 0.5 * len(orden) + 1.5))
plt.barh(orden, intervalos.loc[orden, "test macro-F1"], xerr=err, capsize=4)
plt.xlabel("Macro-F1 en test (IC 95 % bootstrap)")
plt.title("Comparación de modelos, nivel 2")
plt.tight_layout()
plt.savefig(RUTA_FIGURAS + "eval_modelos_nivel2.png", dpi=120)
plt.show()

## 4. Nivel 1 (6 clases): ¿cambia el ranking?

In [ ]:
pred_val1 = cargar_predicciones("nivel1", "val")
pred_test1 = cargar_predicciones("nivel1", "test")

filas = []
for modelo in pred_test1:
    filas.append({"modelo": modelo,
                  "nivel 1 test macro-F1": metricas(pred_test1[modelo])["macro-F1"],
                  "nivel 2 test macro-F1": metricas(pred_test[modelo])["macro-F1"] if modelo in pred_test else np.nan})

tabla_niveles = pd.DataFrame(filas).set_index("modelo").round(3)
tabla_niveles["ranking nivel 1"] = tabla_niveles["nivel 1 test macro-F1"].rank(ascending=False).astype(int)
tabla_niveles["ranking nivel 2"] = tabla_niveles["nivel 2 test macro-F1"].rank(ascending=False)
tabla_niveles.to_csv(RUTA_RESULTADOS + "evaluacion_niveles.csv")
tabla_niveles.sort_values("ranking nivel 1")

## 5. Análisis de errores del mejor modelo

### 5.1. F1 por clase

In [ ]:
mejor_df = pred_test[mejor]
informe = pd.DataFrame(classification_report(mejor_df["y_true"], mejor_df["y_pred"],
                                             output_dict=True, zero_division=0)).T
por_clase = informe.drop(["accuracy", "macro avg", "weighted avg"]).sort_values("f1-score")
por_clase["support"] = por_clase["support"].astype(int)

plt.figure(figsize=(8, 10))
plt.barh(por_clase.index, por_clase["f1-score"])
for i, n in enumerate(por_clase["support"]):
    plt.text(por_clase["f1-score"].iloc[i] + 0.01, i, f"n={n}", va="center", fontsize=7)
plt.xlabel("F1 en test")
plt.title(f"F1 por clase ({mejor}, nivel 2)")
plt.xlim(0, 1.1)
plt.tight_layout()
plt.savefig(RUTA_FIGURAS + "eval_f1_por_clase.png", dpi=120)
plt.show()

por_clase.round(3)

### 5.2. Matriz de confusión

Normalizada por filas: cada fila suma 1 y dice a qué clases se van los proyectos de esa clase real.
Ordenamos las clases por su categoría de nivel 1, así las categorías hermanas quedan juntas y las
confusiones "razonables" aparecen como bloques cerca de la diagonal.

In [ ]:
clases = sorted(madre, key=lambda c: (madre[c], c))   # ordenadas por nivel 1 y luego por nombre
cm = confusion_matrix(mejor_df["y_true"], mejor_df["y_pred"], labels=clases, normalize="true")

plt.figure(figsize=(11, 10))
plt.imshow(cm, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(fraction=0.046)
plt.xticks(range(len(clases)), clases, rotation=90, fontsize=7)
plt.yticks(range(len(clases)), clases, fontsize=7)
plt.xlabel("Predicha")
plt.ylabel("Real")

# Líneas que separan los bloques de nivel 1
limites = [i for i in range(1, len(clases)) if madre[clases[i]] != madre[clases[i - 1]]]
for l in limites:
    plt.axhline(l - 0.5, color="red", linewidth=0.8)
    plt.axvline(l - 0.5, color="red", linewidth=0.8)

plt.title(f"Matriz de confusión normalizada ({mejor}, nivel 2)")
plt.tight_layout()
plt.savefig(RUTA_FIGURAS + "eval_matriz_confusion.png", dpi=120)
plt.show()

### 5.3. ¿Dónde fallan los modelos?

Clasificamos cada error en tres tipos:
- **Categoría descartada**: el modelo predice otra de las áreas a las que el proyecto también pertenece
  (sección 3 del enunciado). No es un fallo "real": es la ambigüedad de quedarnos con una sola etiqueta.
- **Hermana**: la predicción no es una descartada, pero es de la misma categoría de nivel 1
  (por ejemplo, *basic medicine* frente a *clinical medicine*). Es un error razonable.
- **No relacionada**: ni descartada ni hermana. Este es el fallo de verdad.

In [ ]:
def tipos_de_error(df):
    errores = df[df["y_true"] != df["y_pred"]]
    tipos = []
    for _, fila in errores.iterrows():
        if fila["y_pred"] in descartadas[fila["projectID"]]:
            tipos.append("descartada")
        elif madre[fila["y_pred"]] == madre[fila["y_true"]]:
            tipos.append("hermana")
        else:
            tipos.append("no relacionada")
    tipos = pd.Series(tipos, dtype=object)
    return {"errores": len(errores),
            "% descartada": 100 * (tipos == "descartada").mean(),
            "% hermana": 100 * (tipos == "hermana").mean(),
            "% no relacionada": 100 * (tipos == "no relacionada").mean()}

tabla_errores = pd.DataFrame({m: tipos_de_error(df) for m, df in pred_test.items() if m != "baseline"}).T.round(1)
tabla_errores.to_csv(RUTA_RESULTADOS + "evaluacion_tipos_error.csv")
tabla_errores

In [ ]:
# Las 10 confusiones más frecuentes del mejor modelo
errores = mejor_df[mejor_df["y_true"] != mejor_df["y_pred"]]
pares = errores.groupby(["y_true", "y_pred"]).size().sort_values(ascending=False).head(10)
pares = pares.reset_index(name="proyectos")
pares["¿hermanas?"] = ["sí" if madre[a] == madre[b] else "no" for a, b in zip(pares["y_true"], pares["y_pred"])]
pares

## 6. Conclusiones de la evaluación

*(Se escriben cuando estén todos los modelos.)*